# Video Pipeline Setup — YouTube-VOS (Phase 2)

Runs in Google Colab. Heavy work here — indexing thousands of videos,
decoding frames at scale, computing full-split statistics — does not
belong on a laptop.

This notebook calls the `evat` package (`evat.data.datasets.youtube_vos`,
`evat.video.*`) — it does not reimplement pipeline logic. See
`docs/datasets.md` ('Selected Video Dataset') for the verified YouTube-VOS
license (non-commercial research use only) before using this dataset for
anything beyond that.

Access requires agreeing to the official YouTube-VOS Terms of Use; this
notebook does not bypass that.

In [ ]:
%pip install -q -e .

In [ ]:
import os

os.environ["EVAT_DATA_ROOT"] = "/content/data"  # example only; set to the real path

In [ ]:
# Validate structure and build the video index (no image content read yet).
!python scripts/prepare_youtube_vos.py --root "$EVAT_DATA_ROOT/youtube_vos" --split train

In [ ]:
# Full-split statistics (only meaningful once actually run against the real
# dataset — do not hand-edit these numbers, only record what this cell
# actually prints).
import os
from pathlib import Path

from evat.data.datasets.youtube_vos import build_video_index

root = Path(os.environ["EVAT_DATA_ROOT"]) / "youtube_vos"
videos = build_video_index(root, split="train")

num_videos = len(videos)
num_frames = sum(len(v.frames) for v in videos)
num_objects = sum(len(v.objects) for v in videos)
categories = {obj.category for v in videos for obj in v.objects}

print("videos:", num_videos)
print("frames:", num_frames)
print("objects:", num_objects)
print("categories:", len(categories))

In [ ]:
# Smoke-test the temporal pipeline against one real video before scaling up.
from evat.video.sampling import uniform_frame_indices
from evat.video.sequence import build_temporal_sequence
from evat.video.tensors import load_temporal_sequence

video = videos[0]
indices = uniform_frame_indices(num_frames_total=len(video.frames), num_samples=8)
sequence = build_temporal_sequence(video, indices)
batch = load_temporal_sequence(sequence, dataset_root=root)

print("video_id:", batch.video_id)
print("images shape [T, C, H, W]:", batch.images.shape)
print("object_ids per frame:", batch.object_ids)

## Save results

Record the actual printed statistics into `docs/datasets.md` / a
`results/` summary once this has actually been run, per CLAUDE.md
Section 12. Do not invent numbers here.